In [83]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [84]:
import pandas as pd
import xgboost as xgb
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


from pandas.api.types import is_integer_dtype, is_float_dtype, is_numeric_dtype


In [85]:
df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")

In [86]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [88]:
drop = ['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']

df = df.drop(columns = drop)

In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 75 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Neighborhood   1460 non-null   object 
 12  Condition1     1460 non-null   object 
 13  Condition2     1460 non-null   object 
 14  BldgType       1460 non-null   object 
 15  HouseStyle     1460 non-null   object 
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuil

In [90]:
cat_cols = [c for c in df.columns if df[c].dtype.name in ("object", "category", "string")]

petites_cat = [c for c in cat_cols if df[c].nunique(dropna=True) <= 5]   # < 5 catégories (hors NaN)
grandes_cat = [c for c in cat_cols if df[c].nunique(dropna=True) > 5]   # > 5 catégories (hors NaN)

In [91]:
ordinal_encoder = OrdinalEncoder()
df[grandes_cat] = ordinal_encoder.fit_transform(df[grandes_cat])


In [92]:
OHEncoder = OneHotEncoder(sparse_output=False)
encoded_features = OHEncoder.fit_transform(df[petites_cat])
encoded_df = pd.DataFrame(encoded_features, columns = OHEncoder.get_feature_names_out(petites_cat))
df = pd.concat([df, encoded_df], axis=1).drop(columns = petites_cat)

In [93]:
cols_with_na = [c for c in df.columns if df[c].isna().any()]

In [94]:
imputer = SimpleImputer(strategy='mean')


In [95]:
df[cols_with_na] = imputer.fit_transform(df[cols_with_na])


In [96]:
df.info(verbose = True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 144 columns):
 #    Column             Non-Null Count  Dtype  
---   ------             --------------  -----  
 0    Id                 1460 non-null   int64  
 1    MSSubClass         1460 non-null   int64  
 2    LotFrontage        1460 non-null   float64
 3    LotArea            1460 non-null   int64  
 4    Neighborhood       1460 non-null   float64
 5    Condition1         1460 non-null   float64
 6    Condition2         1460 non-null   float64
 7    HouseStyle         1460 non-null   float64
 8    OverallQual        1460 non-null   int64  
 9    OverallCond        1460 non-null   int64  
 10   YearBuilt          1460 non-null   int64  
 11   YearRemodAdd       1460 non-null   int64  
 12   RoofStyle          1460 non-null   float64
 13   RoofMatl           1460 non-null   float64
 14   Exterior1st        1460 non-null   float64
 15   Exterior2nd        1460 non-null   float64
 16   MasV

In [97]:
y = df['SalePrice']
X = df.drop(columns = 'SalePrice')

In [98]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [101]:
print(y_test)

892     154500
1105    325000
413     115000
522     159000
1036    315500
         ...  
479      89471
1361    260000
802     189000
651     108000
722     124500
Name: SalePrice, Length: 292, dtype: int64


In [99]:
model = xgb.XGBRegressor()
model.fit(X_train, y_train)
y_pred =model.predict(X_test)


In [100]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

26471.244231175657
